# 📷 Reconhecimento de Objetos com Webcam
**Bibliotecas:** OpenCV + MediaPipe (Google)  
**Modelo:** MediaPipe Object Detector (EfficientDet-Lite)

Este notebook detecta objetos em tempo real usando sua webcam, desenhando bounding boxes e labels sobre cada objeto reconhecido.

## 📦 1. Instalação das dependências

Instale com `uv` antes de abrir o notebook:
```bash
uv add opencv-python mediapipe
```

Ou direto na célula:

In [ ]:
# Descomente se precisar instalar no ambiente do notebook
# !pip install opencv-python mediapipe -q

## 📥 2. Download do modelo

O MediaPipe precisa de um arquivo `.tflite` com os pesos do detector.

In [1]:
import urllib.request
import os

MODEL_URL  = "https://storage.googleapis.com/mediapipe-models/object_detector/efficientdet_lite0/int8/1/efficientdet_lite0.tflite"
MODEL_PATH = "efficientdet_lite0.tflite"

if not os.path.exists(MODEL_PATH):
    print("Baixando modelo...")
    urllib.request.urlretrieve(MODEL_URL, MODEL_PATH)
    print(f"✅ Modelo salvo em: {MODEL_PATH}")
else:
    print(f"✅ Modelo já existe: {MODEL_PATH}")

Baixando modelo...
✅ Modelo salvo em: efficientdet_lite0.tflite


## 📚 3. Importações

In [2]:
import cv2
import mediapipe as mp
from mediapipe.tasks import python as mp_python
from mediapipe.tasks.python import vision as mp_vision
from mediapipe.tasks.python.components import containers
import numpy as np

## ⚙️ 4. Configurações

In [3]:
# ── Índice da webcam (0 = câmera padrão) ─────────────────────────────────────
CAMERA_INDEX = 0

# ── Score mínimo de confiança para exibir uma detecção (0.0 – 1.0) ───────────
SCORE_THRESHOLD = 0.4

# ── Número máximo de objetos detectados por frame ────────────────────────────
MAX_RESULTS = 5

# ── Cores das bounding boxes (BGR) ───────────────────────────────────────────
COR_BOX    = (0, 255, 120)   # verde
COR_TEXTO  = (255, 255, 255) # branco
COR_FUNDO  = (0, 180, 80)    # fundo da label

# ── Tecla para encerrar (pressione durante a janela do OpenCV) ────────────────
TECLA_SAIR = "q"

print("Configurações carregadas.")

Configurações carregadas.


## 🤖 5. Inicialização do Detector

In [4]:
base_options = mp_python.BaseOptions(model_asset_path=MODEL_PATH)

options = mp_vision.ObjectDetectorOptions(
    base_options=base_options,
    running_mode=mp_vision.RunningMode.IMAGE,
    score_threshold=SCORE_THRESHOLD,
    max_results=MAX_RESULTS,
)

detector = mp_vision.ObjectDetector.create_from_options(options)
print("✅ Detector pronto!")

✅ Detector pronto!


## 🔍 6. Funções Auxiliares

In [5]:
def desenhar_deteccoes(frame, resultado):
    """Desenha bounding boxes e labels sobre o frame."""
    h, w = frame.shape[:2]

    for deteccao in resultado.detections:
        bbox = deteccao.bounding_box
        x1 = int(bbox.origin_x)
        y1 = int(bbox.origin_y)
        x2 = int(bbox.origin_x + bbox.width)
        y2 = int(bbox.origin_y + bbox.height)

        # Bounding box
        cv2.rectangle(frame, (x1, y1), (x2, y2), COR_BOX, 2)

        # Label + score
        categoria = deteccao.categories[0]
        label     = categoria.category_name
        score     = categoria.score
        texto     = f"{label}: {score:.0%}"

        # Fundo da label
        (tw, th), _ = cv2.getTextSize(texto, cv2.FONT_HERSHEY_SIMPLEX, 0.55, 1)
        cv2.rectangle(frame, (x1, y1 - th - 8), (x1 + tw + 6, y1), COR_FUNDO, -1)

        # Texto
        cv2.putText(
            frame, texto,
            (x1 + 3, y1 - 5),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.55, COR_TEXTO, 1, cv2.LINE_AA
        )

    return frame


def frame_para_mp(frame_bgr):
    """Converte frame BGR do OpenCV para MediaPipe Image."""
    frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
    return mp.Image(image_format=mp.ImageFormat.SRGB, data=frame_rgb)


print("Funções carregadas.")

Funções carregadas.


## 🚀 7. Loop Principal — Detecção em Tempo Real

> **Para encerrar:** pressione `q` na janela do OpenCV.

In [8]:
cap = cv2.VideoCapture(CAMERA_INDEX)

if not cap.isOpened():
    print(f"❌ Não foi possível abrir a câmera (índice {CAMERA_INDEX}).")
    print("   Tente mudar CAMERA_INDEX para 1 ou 2 nas configurações.")
else:
    print("✅ Câmera aberta. Pressione 'q' na janela para encerrar.")

    while True:
        ret, frame = cap.read()

        if not ret:
            print("⚠️  Falha ao capturar frame.")
            break

        # Detecção
        mp_image   = frame_para_mp(frame)
        resultado  = detector.detect(mp_image)

        # Desenha resultados
        frame_anotado = desenhar_deteccoes(frame.copy(), resultado)

        # Contador de objetos no canto
        n_obj = len(resultado.detections)
        cv2.putText(
            frame_anotado,
            f"Objetos: {n_obj}",
            (10, 28),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.8, (255, 255, 0), 2, cv2.LINE_AA
        )

        cv2.imshow("Reconhecimento de Objetos — pressione Q para sair", frame_anotado)

        if cv2.waitKey(1) & 0xFF == ord(TECLA_SAIR):
            print("Encerrando...")
            break

    cap.release()
    cv2.destroyAllWindows()
    print("✅ Câmera liberada.")

✅ Câmera aberta. Pressione 'q' na janela para encerrar.
Encerrando...
✅ Câmera liberada.


## 📸 8. Captura de Foto Única (opcional)

Detecta objetos em um único frame e exibe o resultado sem abrir o loop de vídeo.

In [ ]:
import matplotlib.pyplot as plt

cap = cv2.VideoCapture(CAMERA_INDEX)
ret, frame = cap.read()
cap.release()

if ret:
    mp_image      = frame_para_mp(frame)
    resultado     = detector.detect(mp_image)
    frame_anotado = desenhar_deteccoes(frame.copy(), resultado)

    # Exibe inline no notebook
    plt.figure(figsize=(10, 6))
    plt.imshow(cv2.cvtColor(frame_anotado, cv2.COLOR_BGR2RGB))
    plt.axis("off")
    plt.title(f"{len(resultado.detections)} objeto(s) detectado(s)", fontsize=14)
    plt.tight_layout()
    plt.show()

    # Lista os objetos no terminal
    print("\nObjetos detectados:")
    for d in resultado.detections:
        cat = d.categories[0]
        print(f"  • {cat.category_name}: {cat.score:.0%}")
else:
    print("❌ Não foi possível capturar o frame.")